# VQE for Lithium Hydride (LiH)

Scale VQE past hydrogen by carving out an active space, then watch ansatz choice decide whether you reach chemical accuracy.

**Objectives:**
- Build the LiH active-space Hamiltonian (2 electrons, 3 active orbitals -> 6 qubits) with PennyLane's differentiable Hartree-Fock backend (no PySCF)
- Find the exact ground energy (FCI) by diagonalizing the qubit Hamiltonian, and the Hartree-Fock reference for comparison
- Run a chemistry-inspired AllSinglesDoubles VQE and reach within 1.6 mHa of FCI
- Contrast with a hardware-efficient ansatz that converges far worse, learning the active-space + ansatz-choice lesson
- Plot energy vs iteration for both ansatze against the FCI and HF reference lines

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: Run this cell first.
# Runs locally on PennyLane's default.qubit simulator -- no AWS, no PySCF.
# Requires: pip install -e '.[full]' from the project root (see `make setup`).
# Uses the differentiable Hartree-Fock backend (method="dhf"), so openfermionpyscf is NOT needed.

import pennylane as qml
from pennylane import numpy as pnp
import numpy as np
import matplotlib.pyplot as plt

# Chemical accuracy: the error tolerance below which energy predictions are
# considered "chemically useful" (~1 kcal/mol).
CHEMICAL_ACCURACY_HA = 1.6e-3  # Hartree (1.6 mHa)

print("PennyLane", qml.__version__)
print(f"Chemical accuracy target: {CHEMICAL_ACCURACY_HA*1000:.1f} mHa")

## 1. The molecule as an operator

LiH in a minimal basis has 12 spin-orbitals -- too many to diagonalize comfortably and more than we need. The chemistry lives near the Fermi level, so we freeze the lithium core and keep an **active space** of 2 electrons in 3 spatial orbitals. That window maps to 6 qubits (one per active spin-orbital).

The geometry is the equilibrium Li-H separation, 2.969 Bohr, with `requires_grad=False` because we are optimizing the circuit parameters, not the nuclear coordinates. We build the qubit Hamiltonian with `method="dhf"` -- PennyLane's own differentiable Hartree-Fock -- which needs no external quantum-chemistry package.

In [ ]:
# LiH geometry in Bohr (nuclei pinned in place for the electronic-structure problem).
geom = pnp.array([[0.0, 0.0, 0.0],
                  [0.0, 0.0, 2.969]], requires_grad=False)

# Active-space Hamiltonian: 2 active electrons, 3 active spatial orbitals -> 6 qubits.
H, nq = qml.qchem.molecular_hamiltonian(
    ["Li", "H"], geom,
    active_electrons=2,
    active_orbitals=3,
    method="dhf",            # differentiable Hartree-Fock; no PySCF required
)

coeffs, ops = H.terms()
print(f"Active-space qubit Hamiltonian for LiH")
print(f"  qubits      : {nq}")
print(f"  Pauli terms : {len(coeffs)}")
print(f"  This 6-qubit operator IS the molecule -- its lowest eigenvalue is the ground energy.")

## 2. The exact answer: FCI by diagonalization

At 6 qubits the Hamiltonian is a 64x64 matrix, small enough to diagonalize directly. Its smallest eigenvalue is the **full configuration interaction (FCI)** energy -- the exact ground-state energy *within this active space*. This is the floor the variational principle promises VQE can only approach from above, never cross.

We also compute the **Hartree-Fock** reference: the energy of the bare occupation state with no electron correlation. The gap between HF and FCI is the correlation energy VQE must recover.

In [ ]:
# FCI: smallest eigenvalue of the dense Hamiltonian matrix.
Hmat = qml.matrix(H, wire_order=range(nq))
fci_energy = float(np.linalg.eigvalsh(Hmat)[0])

# Hartree-Fock reference state: lowest 2 spin-orbitals occupied -> |1 1 0 0 0 0>.
hf = qml.qchem.hf_state(electrons=2, orbitals=nq)
print("Hartree-Fock occupation (ASCII ket):  |" + " ".join(map(str, hf)) + ">")

# HF energy = expectation of H in the bare reference state (no excitations applied).
dev = qml.device("default.qubit", wires=nq)

@qml.qnode(dev)
def hf_cost():
    qml.BasisState(hf, wires=range(nq))
    return qml.expval(H)

hf_energy = float(hf_cost())

print(f"\nFCI energy (exact, active space): {fci_energy:.6f} Ha")
print(f"Hartree-Fock energy            : {hf_energy:.6f} Ha")
print(f"Correlation energy (HF - FCI)  : {(hf_energy - fci_energy)*1000:.2f} mHa")
print("VQE's job is to recover that correlation gap.")

## 3. The chemistry-inspired ansatz: AllSinglesDoubles

The right ansatz knows chemistry. Starting from the Hartree-Fock state, we apply every single and double **excitation** allowed in this active space -- promoting one or two electrons into virtual orbitals. This is the structure of unitary coupled cluster (UCCSD), and `AllSinglesDoubles` wires it up for us with one parameter per excitation.

Because each parameter corresponds to a real physical excitation, the optimizer has a chemically meaningful landscape to descend. We start every parameter at zero -- that *is* the Hartree-Fock state -- so the first measured energy is exactly the HF energy, and the optimizer pulls it down toward FCI.

In [ ]:
# Enumerate the single and double excitations out of the HF state.
singles, doubles = qml.qchem.excitations(electrons=2, orbitals=nq)
n_params = len(singles) + len(doubles)
print(f"singles: {len(singles)}   doubles: {len(doubles)}   total parameters: {n_params}")

@qml.qnode(dev)
def asd_cost(params):
    qml.AllSinglesDoubles(params, wires=range(nq),
                          hf_state=hf, singles=singles, doubles=doubles)
    return qml.expval(H)

# Starting all-zeros reproduces the Hartree-Fock state.
params = pnp.zeros(n_params, requires_grad=True)
print(f"Energy at theta = 0 (should equal HF): {float(asd_cost(params)):.6f} Ha")

In [ ]:
# Optimize with vanilla gradient descent. ~60 steps is plenty for 6 qubits.
opt = qml.GradientDescentOptimizer(stepsize=0.4)
N_STEPS = 60

asd_history = [float(asd_cost(params))]
for _ in range(N_STEPS):
    params, energy = opt.step_and_cost(asd_cost, params)
    asd_history.append(float(energy))

asd_final = float(asd_cost(params))
asd_gap_mha = (asd_final - fci_energy) * 1000

print(f"AllSinglesDoubles final energy : {asd_final:.6f} Ha")
print(f"FCI energy                     : {fci_energy:.6f} Ha")
print(f"Gap to FCI                     : {asd_gap_mha:.4f} mHa")

# Assertion: the chemistry-inspired ansatz must reach chemical accuracy.
assert asd_gap_mha < CHEMICAL_ACCURACY_HA * 1000, "AllSinglesDoubles missed chemical accuracy"
print(f"\nPASS: within {CHEMICAL_ACCURACY_HA*1000:.1f} mHa of FCI -- chemical accuracy reached.")

## 4. The contrast: a hardware-efficient ansatz

A **hardware-efficient ansatz (HEA)** throws away chemical meaning for shallowness: just layers of generic single-qubit rotations and entanglers, the gates a device happens to support. It runs anywhere, but it does not know about electron occupation or excitations -- so its optimizer wanders a landscape with no chemical signposts.

We give it a fair shot: start from the same Hartree-Fock state, use 3 layers of `StronglyEntanglingLayers`, the same optimizer, and the same 60 steps. Watch where it lands.

In [ ]:
@qml.qnode(dev)
def hea_cost(weights):
    qml.BasisState(hf, wires=range(nq))                 # same HF starting point
    qml.StronglyEntanglingLayers(weights, wires=range(nq))
    return qml.expval(H)

hea_shape = qml.StronglyEntanglingLayers.shape(n_layers=3, n_wires=nq)
print(f"HEA weight tensor shape: {hea_shape}  ({int(np.prod(hea_shape))} parameters)")

# Small random initialization near the HF state.
rng = np.random.default_rng(42)
hea_weights = pnp.array(rng.normal(0.0, 0.1, hea_shape), requires_grad=True)

opt_hea = qml.GradientDescentOptimizer(stepsize=0.4)
hea_history = [float(hea_cost(hea_weights))]
for _ in range(N_STEPS):
    hea_weights, energy = opt_hea.step_and_cost(hea_cost, hea_weights)
    hea_history.append(float(energy))

hea_final = float(hea_cost(hea_weights))
hea_gap_mha = (hea_final - fci_energy) * 1000

print(f"\nHardware-efficient final energy : {hea_final:.6f} Ha")
print(f"Gap to FCI                      : {hea_gap_mha:.1f} mHa")
print(f"AllSinglesDoubles gap, for scale: {asd_gap_mha:.4f} mHa")

## 5. The lesson, quantified

The two numbers are not close. With *more* parameters, the hardware-efficient ansatz stalls hundreds of millihartree above FCI -- worse, in fact, than plain Hartree-Fock -- while AllSinglesDoubles, with a handful of chemically motivated parameters, lands essentially on the exact answer.

This is the central NISQ trade-off made concrete: a shallow, generic circuit is easy to run but hard to train toward the *right* state, because its parameters carry no chemical structure. The active-space choice shrank the problem to something tractable; the ansatz choice decided whether "tractable" became "correct."

In [ ]:
print(f"{'ansatz':<22}{'final energy (Ha)':>20}{'gap to FCI (mHa)':>20}")
print("-" * 62)
print(f"{'Hartree-Fock (ref)':<22}{hf_energy:>20.6f}{(hf_energy-fci_energy)*1000:>20.2f}")
print(f"{'AllSinglesDoubles':<22}{asd_final:>20.6f}{asd_gap_mha:>20.4f}")
print(f"{'HardwareEfficient':<22}{hea_final:>20.6f}{hea_gap_mha:>20.2f}")
print("-" * 62)
print(f"{'FCI (exact)':<22}{fci_energy:>20.6f}{0.0:>20.2f}")

# The chemistry-inspired ansatz should beat the hardware-efficient one by a wide margin.
assert asd_gap_mha < hea_gap_mha, "expected AllSinglesDoubles to converge better than HEA"
print("\nPASS: AllSinglesDoubles converges far closer to FCI than the hardware-efficient ansatz.")

## 6. Convergence: energy vs iteration

One picture tells the whole story. We plot both convergence curves against two horizontal references: the **FCI** line (the exact floor) and the **Hartree-Fock** line (the no-correlation starting point). AllSinglesDoubles should peel off HF and dive onto FCI; the hardware-efficient curve should flatten out well above both.

In [ ]:
iters_asd = range(len(asd_history))
iters_hea = range(len(hea_history))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(iters_asd, asd_history, "-o", ms=3, lw=1.6, color="#2563eb",
        label="AllSinglesDoubles (chemistry-inspired)")
ax.plot(iters_hea, hea_history, "-s", ms=3, lw=1.6, color="#dc2626",
        label="StronglyEntanglingLayers (hardware-efficient)")
ax.axhline(fci_energy, ls="--", lw=1.4, color="#16a34a", label=f"FCI (exact) = {fci_energy:.4f} Ha")
ax.axhline(hf_energy, ls=":", lw=1.4, color="#6b7280", label=f"Hartree-Fock = {hf_energy:.4f} Ha")

ax.set_xlabel("optimization step")
ax.set_ylabel("energy (Ha)")
ax.set_title("LiH VQE convergence: ansatz choice decides the outcome")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("AllSinglesDoubles tracks down onto the FCI line; the hardware-efficient curve stalls above HF.")

## Exercises

Work through these to deepen the active-space and ansatz intuition. All run locally on `default.qubit` -- no AWS.

In [ ]:
# Exercise 1: Enlarge the active space.
# Rebuild H with active_orbitals=4 (-> 8 qubits) and re-find FCI by diagonalization.
# Does AllSinglesDoubles still reach chemical accuracy? How many parameters now?
#
# TODO:
# H4, nq4 = qml.qchem.molecular_hamiltonian(["Li","H"], geom,
#                                           active_electrons=2, active_orbitals=4, method="dhf")
# fci4 = float(np.linalg.eigvalsh(qml.matrix(H4, wire_order=range(nq4)))[0])
# ... build hf4, singles4, doubles4, optimize, compare to fci4 ...

In [ ]:
# Exercise 2: Give the hardware-efficient ansatz more depth.
# Try n_layers = 4, 6, 8 for StronglyEntanglingLayers. Does adding layers close the gap to FCI,
# or do you see the optimization stall (a hint of barren plateaus)?
#
# TODO:
# for L in (4, 6, 8):
#     shape = qml.StronglyEntanglingLayers.shape(n_layers=L, n_wires=nq)
#     ... reinitialize, optimize, record final gap to FCI ...

In [ ]:
# Exercise 3: Sweep the bond length to trace a slice of LiH's potential energy surface.
# Rebuild H at separations from ~2.0 to ~4.5 Bohr, run the AllSinglesDoubles VQE at each,
# and plot energy vs separation. Where is the minimum, and how deep is the well?
#
# TODO:
# separations = pnp.linspace(2.0, 4.5, 8, requires_grad=False)
# energies = []
# for d in separations:
#     g = pnp.array([[0.,0.,0.],[0.,0.,float(d)]], requires_grad=False)
#     ... build H, run VQE, append final energy ...
# plt.plot(separations, energies)

## Summary

- LiH is too big for the full minimal basis, so we kept an **active space** of 2 electrons in 3 orbitals -> a 6-qubit, 62-term Hamiltonian built with PennyLane's `method="dhf"` (no PySCF).
- Diagonalizing that operator gives the exact **FCI** energy (-7.8637 Ha); the **Hartree-Fock** reference sits ~1 mHa above it, and that gap is the correlation energy VQE must recover.
- The chemistry-inspired **AllSinglesDoubles** ansatz reached FCI to within ~0.001 mHa -- comfortably inside the 1.6 mHa chemical-accuracy bar -- in 60 gradient-descent steps.
- A **hardware-efficient** ansatz, despite more parameters, stalled hundreds of mHa above FCI. Active space made the problem tractable; **ansatz choice** made it correct.
- The throughline holds: molecule -> operator -> matrix -> lowest eigenvalue, chased by minimizing an expectation value. Everything else is engineering in service of that one move.

**Next:** [`05-ansatz-design.ipynb`](05-ansatz-design.ipynb) -- compare UCCSD, hardware-efficient, and ADAPT-VQE on a larger active space, measuring circuit depth and CNOT count against accuracy.